In [15]:
import json
import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import login
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch
import re
from tqdm import tqdm
from typing import Optional, List, Tuple
from IPython.display import display, Markdown


# check if gpu is available
device = 0 if torch.cuda.is_available() else -1
print("Using device:", "GPU" if device == 0 else "CPU")

load_dotenv()  
hf_token = os.getenv("HF_TOKEN")
login(token=hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Using device: GPU


In [2]:
MODELS = [
    'meta-llama/Llama-3.2-3B-Instruct',
    'Qwen/Qwen2.5-3B-Instruct',
    # Qwen/Qwen2.5-7B-Instruct,
    # mistralai/Mixtral-8x7B-Instruct-v0.1,
    # meta-llama/Llama-3.1-8B-Instruct,
]

def load_model(model_name: str):
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="cuda" if torch.cuda.is_available() else None,
        torch_dtype="auto",
        trust_remote_code=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # se manca pad token, usiamo eos come pad
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    # impostiamo padding_side='left' per decoder-only mode
    tokenizer.padding_side = "left"

    # Creiamo la pipeline (la pipeline userà il tokenizer modificato)
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        dtype=torch.float16 ,#"auto"
        #torch_dtype=torch.float16 #"auto"
    )

    # Assicuriamoci che la pipeline conosca il pad_token_id (alcune versioni lo richiedono esplicitamente)
    pipe.tokenizer.pad_token_id = tokenizer.pad_token_id

    return pipe, model, tokenizer

def destroy_model(model, tokenizer):
    if model is not None:
        del model
    
    if tokenizer is not None:
        del tokenizer

    torch.cuda.empty_cache()

In [22]:
import re
import json5  # <--- parser robusto
from typing import Optional

def clear_output(text: str, num_turns: int) -> Optional[dict]:
    if not text:
        print("Testo vuoto.")
        return None

    # Estrai il blocco JSON
    match = re.search(r'<JSON\s*>(.*?)</JSON\s*>', text, re.DOTALL | re.IGNORECASE)
    if not match:
        print("Nessun blocco <JSON> trovato.")
        return None

    json_str = match.group(1).strip()

    # Rimuove trailing commas illegali
    json_str = re.sub(r',\s*([\]}])', r'\1', json_str)

    # Parsing robusto con JSON5
    try:
        data = json5.loads(json_str)
    except Exception as e:
        print("Errore nel parsing JSON5:", e)
        print("JSON estratto:\n", json_str)
        return None

    # -------- VALIDAZIONI -------- #

    # Deve contenere solo la chiave "turns"
    if not isinstance(data, dict):
        print("Il JSON non è un oggetto.")
        return None

    keys = list(data.keys())
    if keys != ["turns"]:
        print(f"Chiavi JSON errate: {keys}, attesa solo ['turns']")
        return None

    # turns deve essere una lista
    if not isinstance(data["turns"], list):
        print('"turns" non è una lista.')
        return None

    # Deve avere num_turns + 1 elementi
    expected = num_turns + 1
    actual = len(data["turns"])

    if actual != expected:
        print(f'Numero di turni errato: trovato {actual}, atteso {expected}.')
        return None

    return data


In [25]:
result = """<JSON>
            {
                "turns": [
                    {
                        "u1": "I'm so drunk I can barely stand",
                        "u2": "Pisser. Amusing joke or incident. You're not the first to get hammered and stumble around"
                    },
                    {
                        "u1": "I'm not drunk, I'm just pissed off. I'm so angry I could scream",
                        "u2": "Pissweak or weak as piss. Very weak. You're not even close to being angry, you're just drunk"
                    },
                    {
                        "u1": "I'm not wasting time, I'm just pissing around. I'm taking a break from the stress",
                        "u2": "Pissing around. Wasting time. You're just avoiding the issue, you're not taking a break from anything"
                    }
                ],
            }
            </JSON>

            Note: The first message from u1 is not explicitly stated in the prompt, but based on the context of the conversation, it is assumed to be a message that sets the tone for the conversation, which is that u1 is drunk and possibly joking around. The first message from u1 is therefore "I'm so drunk I can barely stand". The first message from u2 is a response to u1's message, which is a lighthearted and humorous take on the situation"""

cleared_json = clear_output(result, 2)
print(cleared_json)


{'turns': [{'u1': "I'm so drunk I can barely stand", 'u2': "Pisser. Amusing joke or incident. You're not the first to get hammered and stumble around"}, {'u1': "I'm not drunk, I'm just pissed off. I'm so angry I could scream", 'u2': "Pissweak or weak as piss. Very weak. You're not even close to being angry, you're just drunk"}, {'u1': "I'm not wasting time, I'm just pissing around. I'm taking a break from the stress", 'u2': "Pissing around. Wasting time. You're just avoiding the issue, you're not taking a break from anything"}]}


In [5]:
MAP = {
    1: "First",
    2: "Second",
    3: "Third",
    4: "Fourth",
    5: "Fifth"
}

def multipico_prompt(post, reply, num_turns) -> str:
    initial_prompt = f"""
    Given this pair of messages extracted from social media:

    u1: {post}
    u2: {reply}

    Generate the {num_turns} previous dialogue turns (each turn consisting of a message from u1 and a message from u2) that logically lead up to the given messages. Make sure to:
    - Maintain the original writing style of both u1 e u2.
    - Preserve the sentiment and tone of the conversation.
    - Keep the dialogue coherent and consistent with the context of the given messages.

    Return the result in the following JSON format, including both the previous 3 turns and the given messages as the last turn, enclosed in <JSON> tags:

    <JSON>"""

    # turni
    turni_prompt = ""
    for i in range(num_turns):
        turni_prompt += f"""
        {{
            "u1": "{MAP[i+1]} message from u1",
            "u2": "{MAP[i+1]} message from u2"
        }},"""

    final_prompt = f"""
    </JSON>

    Response:
    <JSON>
    {{
        "turns": [
            {{
                "u1": "
    """

    prompt = initial_prompt + turni_prompt + final_prompt

    return prompt

def csc_prompt(context, reply, num_turns, relation) -> str:
    initial_prompt = f"""
    Given the context and the reply of a user u2 to u1 (whose name is Steve and your relation with him is {relation}):

    context: {context},
    u2 : {reply},

    Generate the {num_turns} previous dialogue turns (each turn = one message from u1 followed by one from u2) plus the u1 message that directly precedes the final given u2 line. Make sure to:
    - Maintain the original writing style of u2.
    - Preserve the sentiment and tone of the conversation.
    - Keep the dialogue coherent and consistent with the context of the given messages

    Return the result in the following JSON format, including both the previous 3 turns and the given messages as the last turn, enclosed in <JSON> tags:

    <JSON>"""

    # turni
    turni_prompt = ""
    for i in range(num_turns):
        turni_prompt += f"""
        {{
            "u1": "{MAP[i+1]} message from u1",
            "u2": "{MAP[i+1]} message from u2"
        }},"""

    final_prompt = f"""
    </JSON>

    Response:
    <JSON>
    {{
        "turns": [
            {{
                "u1": "
    """

    prompt = initial_prompt + turni_prompt + final_prompt

    return prompt


#prompt = csc_prompt("hello", "hi", 3, "friend")
#display(Markdown(f"```json\n{prompt}\n```"))


In [6]:
TEMPERATURE = 0.5
BATCH_SIZE = 1
NUM_MAX_TURNS = 3

def generate_batch_output(pipe, tokenizer, turn: List[Tuple[str, str]], dataset_name: str, num_turns: int):
    """
    Genera output in batch per una lista di coppie di messaggi.
    
    Args:
        pipe: il modello di generazione (pipeline o callable)
        tokenizer: tokenizer associato al modello
        turn: lista di tuple (post, reply)
        dataset_name: nome del dataset utilizzato
    
    Returns:
        Lista di tuple (prompt, model_output, dialogue_dict)
    """
    
    prompts = []
    for post, reply, relation in turn:
        if dataset_name == "MultiPICo":
            prompt = multipico_prompt(post, reply, num_turns)
        elif dataset_name == "CSC":
            prompt = csc_prompt(post, reply, num_turns, relation)

        prompts.append(prompt)

    # Genera i risultati dal modello
    outputs = pipe(
        prompts,
        max_new_tokens=1024,
        do_sample=TEMPERATURE > 0,
        temperature=TEMPERATURE,
        return_full_text=False,
        batch_size=BATCH_SIZE,
        pad_token_id=tokenizer.eos_token_id
    )


    results = []
    for prompt, out in zip(prompts, outputs):
        entry = out[0]
        model_output = entry['generated_text'].strip()
        #print("Model output:", model_output)
        initial_output = """<JSON>{"turns": [{"u1":\""""
        dialogue = clear_output(f"{initial_output}{model_output}")
        results.append((prompt, model_output, dialogue))

    return results


In [7]:
def generate_dialogue(dataset_df: pd.DataFrame, dataset_name: str):
    for model_name in MODELS:
        print(f"Processing model: {model_name}")
        pipe, model, tokenizer = load_model(model_name)
        print("Model and tokenizer loaded.")
    
        for i in range(NUM_MAX_TURNS):
            num_turn = i + 1

            output_file = f'{dataset_name}_{model_name.replace("/", "_")}_num_turn_{num_turn}.json'
            output_rows = []

            # --- carica dati già esistenti, se presenti ---
            if os.path.exists(output_file):
                try:
                    with open(output_file, 'r', encoding='utf-8') as f:
                        existing_data = json.load(f)
                    existing_ids = set([row['unique_id'] for row in existing_data])
                    print(f"Found {len(existing_ids)} already processed entries in {output_file}")
                    # aggiungi i dati esistenti a output_rows
                    output_rows.extend(existing_data)
                except Exception as e:
                    print(f"Error loading existing JSON: {e}")
                    existing_ids = set()
            else:
                existing_ids = set()

            # filtra dataset_df per saltare già processati
            dataset_df = dataset_df[~dataset_df['unique_id'].isin(existing_ids)]
            print(f"{len(dataset_df)} new entries to process")

            for start in tqdm(range(0, len(dataset_df), BATCH_SIZE), desc=f"Running {model_name}"):
                batch_df = dataset_df.iloc[start:start+BATCH_SIZE]

                post = batch_df['post'].tolist()
                #post = [re.sub(r'@USER', '', p) for p in post]

                reply = batch_df['reply'].tolist()
                #reply = [re.sub(r'@USER', '', r) for r in reply]

                if dataset_name == "CSC":
                    relation = batch_df['annotations'].apply(lambda x: x['speaker_annotation']['interlocutor_relationship']).tolist()
                else:
                    relation = [None] * len(post)

                turn = list(zip(post, reply, relation))

                pipe, model, tokenizer = load_model(model_name)
                results = generate_batch_output(pipe, tokenizer, turn, dataset_name, num_turn)

                cleaned_results = []
                for i, (prompt, model_output, dialogue) in enumerate(results):
                    invalid_inferences = 0
                    while dialogue is None and invalid_inferences < 3:
                        invalid_inferences += 1
                        prompt, model_output, dialogue = generate_batch_output(pipe, tokenizer, [turn[i]], dataset_name, num_turn)[0]

                    if invalid_inferences >= 3:
                        print(f"⚠️  Warning: Max retries reached for entry {batch_df.iloc[i]['unique_id']}. Saving last output anyway.")
                        dialogue = {"error": "Max retries reached"}
                        model_output = {"error": "Max retries reached"}

                    cleaned_results.append((prompt, model_output, dialogue, invalid_inferences))

                for row, (prompt, model_output, dialogue, invalid_inferences) in zip(batch_df.itertuples(), cleaned_results):
                    enriched = row._asdict()
                    enriched['prompt'] = prompt
                    enriched['model_output'] = model_output
                    enriched['dialogue'] = dialogue
                    # todo: salvare le generazioni non valide
                    enriched['invalid_inferences'] = invalid_inferences
                    output_rows.append(enriched)

                # salva ad ogni batch
                pd.DataFrame(output_rows).to_json(
                    output_file,
                    orient='records',
                    indent=2,
                    force_ascii=False
                )
                print(f"✔ Batch {start//BATCH_SIZE + 1} salvato → {output_file}")

            destroy_model(model, tokenizer)
            print("Model and tokenizer destroyed.")

In [8]:
# check if exists and open unified corpus in 1_datasets_merge\output\unified_corpus.jsonù
file_path = '../1_datasets_merge/output/unified_corpus.json'
try:
    with open(file_path, 'r', encoding='utf-8') as f:
        unified_corpus = json.load(f)
    print(f"Loaded unified corpus with {len(unified_corpus)} entries")
    df = pd.DataFrame(unified_corpus)
except Exception as e:
    print(f"Error loading unified corpus: {e}")
    raise e

# get only row with language == 'en' and dataset == 'MultiPICo'
#df_en_multipico = df[(df['language'] == 'en') & (df['dataset_name'] == 'MultiPICo')]
# show number of entries
#print(f"Filtered MultiPICo English entries: {len(df_en_multipico)}")
#df_en_multipico.head()

datasets = [
    'MultiPICo',
    'CSC',
]

for dataset in datasets:
    print(f"Processing dataset: {dataset}")
    df_filtered = df[(df['language'] == 'en') & (df['dataset_name'] == dataset)]
    print(f"Filtered {dataset} English entries: {len(df_filtered)}")
    generate_dialogue(df_filtered, dataset)

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded unified corpus with 25814 entries
Processing dataset: MultiPICo
Filtered MultiPICo English entries: 2999
Processing model: meta-llama/Llama-3.2-3B-Instruct


Loading checkpoint shards: 100%|██████████| 2/2 [00:12<00:00,  6.13s/it]
Device set to use cuda


Model and tokenizer loaded.
2999 new entries to process


Loading checkpoint shards: 100%|██████████| 2/2 [00:16<00:00,  8.06s/it]<?, ?it/s]
Device set to use cuda
Running meta-llama/Llama-3.2-3B-Instruct:   0%|          | 0/2999 [05:51<?, ?it/s]


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# get only 700 entries for testing
#df_en_multipico = df_en_multipico.head(100)
#print(f"Using {len(df_en_multipico)} entries for testing"
#generate_dialogue(df_en_multipico)

Loaded unified corpus with 25814 entries
Filtered MultiPICo English entries: 2999
Using 100 entries for testing
